## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool ,OpenAIChatCompletionsModel
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push
import os
from openai import AsyncOpenAI


In [ ]:
load_dotenv(override=True)

In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

In [ ]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GOOGLE_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

In [ ]:
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
google_client = AsyncOpenAI(base_url=GOOGLE_BASE_URL,api_key=google_api_key)

In [ ]:
openrouter_model = OpenAIChatCompletionsModel(model="openrouter/free", openai_client=openrouter_client)
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)
google_model = OpenAIChatCompletionsModel(model="gemini-3.5-flash-lite",openai_client=google_client,)

In [ ]:
# Constants 

MODEL_NAME = google_model
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

# In house tools 

## Agent 1: The Search Agent



###  DO NOT USE GROQ OR OPEN ROUTER AS THEY MIGHT CRASH USE GEMINI FREE TIRE !

In [ ]:
%pip install -U ddgs

In [ ]:
from ddgs import DDGS
from agents import function_tool


def search_web(query: str) -> str:
    """Search the web using DuckDuckGo."""

    results = DDGS().text(
        query,
        max_results=5,
    )

    if not results:
        return "No search results found."

    formatted_results = []

    for result in results:
        formatted_results.append(
            f"Title: {result.get('title', 'No title')}\n"
            f"URL: {result.get('href', '')}\n"
            f"Snippet: {result.get('body', '')}"
        )

    return "\n\n---\n\n".join(formatted_results)


web_search = function_tool(search_web, name_override="web_search")


In [ ]:
research_agent = Agent(
    name="Research Agent",

    instructions="""
You are a research assistant.

You have ONE tool available: web_search.

When you need information from the web, ALWAYS use the web_search tool.

Do NOT use a tool named visit.
Do NOT use a tool named browse.
Do NOT use a tool named search.
Do NOT invent or call any other tools.

Use web_search with the search query provided by the user.
After receiving the search results, summarize the useful information.
""",

    model=MODEL_NAME,

    tools=[web_search],
)

<H5> The structure till now : 
DuckDuckGo
    ->
search_web()
    ->
web_search (Agents SDK FunctionTool)
    ->
Agent
    ->
OpenRouter model



In [ ]:
task = "Most popular AI Agent frameworks in 2026"
result = await Runner.run(
    research_agent,
    task
)

print(result.final_output)


## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


### As always, take a look at the trace

https://platform.openai.com/traces

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [ ]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [ ]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant.

Given the user's query, create a research search plan.

You MUST return exactly {HOW_MANY_SEARCHES} search items.

Each search item must contain:
- reason: why this search is useful
- query: the exact web search query

Return ONLY valid JSON matching the WebSearchPlan schema.

Do not return Markdown.
Do not return explanations.
Do not return an empty response.
"""

planner_agent = Agent(
    name="Planner Agent",
    instructions=INSTRUCTIONS,
    model=MODEL_NAME,
    output_type=WebSearchPlan,
)

In [ ]:
task
result = await Runner.run(planner_agent, task)
result.final_output

## Agent 3: The Writer Agent

In [ ]:
class ReportData(BaseModel):
    short_summary: str = Field(
        description="A short 2-3 sentence summary of the findings."
    )
    markdown_report: str = Field(
        description="The final report"
    )
    follow_up_questions: list[str] = Field(
        description="Suggested topics to research further"
    )

In [ ]:
WRITER_INSTRUCTIONS = """
You are a research report writer.

You will receive:
1. The original user query.
2. Summarized web search results.

Write a comprehensive, accurate research report based ONLY on the provided
research information.

Your output MUST conform exactly to the ReportData schema.

The output must contain exactly these three fields:
- short_summary: A concise 2-3 sentence summary.
- markdown_report: The complete report written in Markdown.
- follow_up_questions: A list of useful questions for further research.

IMPORTANT:
Return ONLY valid JSON.
Do NOT return Markdown directly.
Do NOT include ```json fences.
Do NOT include any text before or after the JSON object.

The markdown_report field itself may contain Markdown.
"""

writer_agent = Agent(
    name="Writer Agent",
    instructions=WRITER_INSTRUCTIONS,
    model=MODEL_NAME,
    output_type=ReportData,
)

<h5> the strcuture :                    OpenRouter free model
                       ->
                  Planner Agent
                      ->
                    Search Plan
                      ->
                  DDGS / DuckDuckGo
                       ->
                  Search Results
                      ->
                  Writer Agent
                       ->
                  Markdown text
                      ->
              Python creates ReportData
                      ->
                    Email report

## Agent 4: The email agent

In [ ]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [ ]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [ ]:
async def run_searches(query: str):
    print("Planning searches...")
    result = None
    for attempt in range(3):
        try:
            result = await Runner.run(planner_agent, f"Query: {query}")
            break
        except Exception as e:
            print(f"Planner attempt {attempt + 1}/3 failed: {e}")
            if attempt == 2:
                raise
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [perform_search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def perform_search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(research_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [ ]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

In [ ]:
print(ReportData.model_json_schema())


### Showtime!

In [ ]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

### As always, take a look at the trace

https://platform.openai.com/traces